# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dyajaballh8/FlyRank_Intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule

I rank content items that have high search visibility but low CTR.

The rule gives higher priority to content with:

- Higher GSC impressions.
- A good average search position.
- Low CTR compared with other content items.

The recommended action is to review the title, meta description, and search-result snippet to improve click-through performance.

### Reason code

The rule outputs one reason code:

`HIGH_IMPRESSIONS_LOW_CTR`

This means that the content item receives many search impressions but has a relatively low CTR, making it a candidate for CTR improvement.

### Action label

`REVIEW_CTR`

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_auth (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
)
""")

DATA_PATH = """
hf://datasets/FlyRank/internship-warehouse/
fact_content_daily_performance/month=2026-05/data_0.parquet
""".replace("\n", "")

# Signal 1: search visibility
signal_1 = con.execute(f"""
SELECT
    CASE
        WHEN gsc_impressions >= 500 THEN 'HIGH'
        WHEN gsc_impressions >= 100 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS impressions_bucket,
    COUNT(*) AS n
FROM read_parquet('{DATA_PATH}')
WHERE gsc_data_available = TRUE
GROUP BY 1
ORDER BY
    CASE impressions_bucket
        WHEN 'HIGH' THEN 1
        WHEN 'MEDIUM' THEN 2
        ELSE 3
    END
""").fetchdf()

print("Signal 1 — Search visibility")
display(signal_1)

# Signal 2: CTR among pages with useful visibility
signal_2 = con.execute(f"""
SELECT
    CASE
        WHEN gsc_impressions >= 500
             AND gsc_avg_position > 0
             AND gsc_avg_position <= 20
             AND gsc_clicks * 100.0 / NULLIF(gsc_impressions, 0) < 0.5
            THEN 'LOW_CTR_OPPORTUNITY'

        WHEN gsc_impressions >= 500
             AND gsc_avg_position > 0
             AND gsc_avg_position <= 20
            THEN 'OTHER_PAGE_ONE_TO_TWENTY'

        ELSE 'OTHER'
    END AS ctr_position_bucket,
    COUNT(*) AS n
FROM read_parquet('{DATA_PATH}')
WHERE gsc_data_available = TRUE
GROUP BY 1
ORDER BY n DESC
""").fetchdf()

print("\nSignal 2 — CTR vs position")
display(signal_2)

print("\nSignal verdicts:")
print("Signal 1 — CONFIRMED")
print("Signal 2 — CONFIRMED")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 — Search visibility


,impressions_bucket,n
0,HIGH,97611
1,MEDIUM,455216
2,LOW,3820595


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Signal 2 — CTR vs position


,ctr_position_bucket,n
0,OTHER,4285109
1,LOW_CTR_OPPORTUNITY,67662
2,OTHER_PAGE_ONE_TO_TWENTY,20651



Signal verdicts:
Signal 1 — CONFIRMED
Signal 2 — CONFIRMED


## 2. Build the ranked queue (writes the CSV)

## Ranked queue

The score combines search visibility, search position, and CTR opportunity.

Higher scores indicate stronger candidates for CTR review.

Only observable information from the decision window is used.
No product flags or future labels are used.

In [2]:
import os
os.makedirs("work/outputs", exist_ok=True)

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue_query = f"""
WITH base AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        gsc_clicks * 100.0 /
            NULLIF(gsc_impressions, 0) AS ctr

    FROM read_parquet('{DATA_PATH}')

    WHERE gsc_data_available = TRUE
      AND gsc_impressions > 0
      AND gsc_avg_position > 0
),

scored AS (
    SELECT
        *,

        CASE
            WHEN gsc_impressions >= 500
             AND gsc_avg_position <= 20
             AND ctr < 0.5
            THEN
                50
                + LEAST(gsc_impressions / 1000.0, 30)
                + LEAST((20 - gsc_avg_position) * 0.5, 10)
                + LEAST((0.5 - ctr) * 20, 10)
            ELSE 0
        END AS score,

        CASE
            WHEN gsc_impressions >= 500
             AND gsc_avg_position <= 20
             AND ctr < 0.5
            THEN 'HIGH_IMPRESSIONS_LOW_CTR'
            ELSE 'NO_PRIORITY'
        END AS reason_code,

        CASE
            WHEN gsc_impressions >= 500
             AND gsc_avg_position <= 20
             AND ctr < 0.5
            THEN 'REVIEW_CTR'
            ELSE 'NO_ACTION'
        END AS action

    FROM base
)

SELECT
    ROW_NUMBER() OVER (
        ORDER BY score DESC, gsc_impressions DESC
    ) AS rank,

    report_date,
    client_hash_id,
    content_hash_id,
    ROUND(score, 2) AS score,
    reason_code,
    action,
    gsc_impressions,
    gsc_clicks,
    ROUND(ctr, 3) AS ctr,
    ROUND(gsc_avg_position, 2) AS avg_position

FROM scored

ORDER BY score DESC, gsc_impressions DESC
"""

queue = con.execute(queue_query).fetchdf()

# Keep ranked queue
queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Queue created successfully.")
print(f"Rows: {len(queue):,}")

display(queue.head(20))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue created successfully.
Rows: 4,237,020


,rank,report_date,client_hash_id,content_hash_id,score,reason_code,action,gsc_impressions,gsc_clicks,ctr,avg_position
0,1,2026-05-17,client_23a62021009f63c4,content_65c75874a23fca87,95.66,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR,32168,2,0.006,8.43
1,2,2026-05-25,client_e547b89c05043229,content_21309e9a83c83653,94.96,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR,46325,58,0.125,5.07
2,3,2026-05-26,client_e547b89c05043229,content_cedadfe5ae4845ac,94.79,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR,47607,57,0.120,5.63
3,4,2026-05-25,client_e547b89c05043229,content_2f567cb6ad16f6f0,93.67,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR,28431,3,0.011,9.10
4,5,2026-05-24,client_e547b89c05043229,content_21309e9a83c83653,91.28,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR,25806,25,0.097,5.18
5,6,2026-05-18,client_23a62021009f63c4,content_65c75874a23fca87,89.75,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR,24391,2,0.008,8.96
6,7,2026-05-26,client_e547b89c05043229,content_545bb6cc7081ded3,89.29,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR,26663,84,0.315,2.15
7,8,2026-05-26,client_e547b89c05043229,content_21309e9a83c83653,88.81,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR,24429,38,0.156,5.01
8,9,2026-05-22,client_157ffe4d4a595515,content_9648c4d1595a0794,88.15,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR,18856,2,0.011,0.99
9,10,2026-05-20,client_23a62021009f63c4,content_65c75874a23fca87,86.53,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CTR,21323,4,0.019,8.84


## 3. Top-20 review

## Weak picks + leakage check

Some high-ranked pages may still be weak recommendations because the rule only observes search visibility, position, and CTR.

Possible failure cases include:
- Different search intent.
- Brand or navigational queries.
- A CTR that is naturally low for the query type.
- Recent changes that are not visible in the decision window.

The rule does not use product decision flags or future-window labels.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

top20["confidence_note"] = (
    "Observed high impressions, position <= 20, and CTR < 0.5."
)

top20["what_would_make_it_wrong"] = (
    "Search intent may not match the page, or the low CTR may be expected "
    "for this query type."
)

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(top20_review)


,rank,content_hash_id,action,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,content_65c75874a23fca87,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,95.66,"Observed high impressions, position <= 20, and...","Search intent may not match the page, or the l..."
1,2,content_21309e9a83c83653,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,94.96,"Observed high impressions, position <= 20, and...","Search intent may not match the page, or the l..."
2,3,content_cedadfe5ae4845ac,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,94.79,"Observed high impressions, position <= 20, and...","Search intent may not match the page, or the l..."
3,4,content_2f567cb6ad16f6f0,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,93.67,"Observed high impressions, position <= 20, and...","Search intent may not match the page, or the l..."
4,5,content_21309e9a83c83653,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,91.28,"Observed high impressions, position <= 20, and...","Search intent may not match the page, or the l..."
5,6,content_65c75874a23fca87,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,89.75,"Observed high impressions, position <= 20, and...","Search intent may not match the page, or the l..."
6,7,content_545bb6cc7081ded3,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,89.29,"Observed high impressions, position <= 20, and...","Search intent may not match the page, or the l..."
7,8,content_21309e9a83c83653,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,88.81,"Observed high impressions, position <= 20, and...","Search intent may not match the page, or the l..."
8,9,content_9648c4d1595a0794,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,88.15,"Observed high impressions, position <= 20, and...","Search intent may not match the page, or the l..."
9,10,content_65c75874a23fca87,REVIEW_CTR,HIGH_IMPRESSIONS_LOW_CTR,86.53,"Observed high impressions, position <= 20, and...","Search intent may not match the page, or the l..."


## 4. Weak picks + leakage check

## Weak picks + leakage check

Some high-ranked pages may still be weak recommendations because the rule only observes search visibility, position, and CTR.

Possible failure cases include:
- Different search intent.
- Brand or navigational queries.
- A CTR that is naturally low for the query type.
- Recent changes that are not visible in the decision window.

The rule does not use product decision flags or future-window labels.

In [5]:
# 4. Weak picks + leakage check

print("Weak picks among selected recommendations:")

weak_picks = queue.tail(10)

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "score",
            "gsc_impressions",
            "ctr",
            "avg_position",
            "reason_code",
            "action"
        ]
    ]
)

# Leakage check
allowed_columns = {
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "avg_position",
    "ctr",
    "score",
    "reason_code",
    "action",
    "rank"
}

used_columns = set(queue.columns)
unexpected = used_columns - allowed_columns

print("\nLeakage check:")
print("No product flags used.")
print("No future label used.")
print("Only observable search-performance fields were used.")

print("\nUnexpected columns:", unexpected)

assert len(unexpected) == 0

print("Leakage check PASSED.")

Weak picks among selected recommendations:


,rank,content_hash_id,score,gsc_impressions,ctr,avg_position,reason_code,action
4237010,4237011,content_d85bd2cee1dcda9b,0.0,1,0.0,1.0,NO_PRIORITY,NO_ACTION
4237011,4237012,content_95aa67878e86dd3f,0.0,1,0.0,5.0,NO_PRIORITY,NO_ACTION
4237012,4237013,content_89b978f66306c175,0.0,1,100.0,10.0,NO_PRIORITY,NO_ACTION
4237013,4237014,content_a9cb1bcc02de90e9,0.0,1,0.0,4.0,NO_PRIORITY,NO_ACTION
4237014,4237015,content_b4f3e4b02ea7dc72,0.0,1,0.0,5.0,NO_PRIORITY,NO_ACTION
4237015,4237016,content_28b6633d2f4e17a2,0.0,1,0.0,2.0,NO_PRIORITY,NO_ACTION
4237016,4237017,content_caf756e63ca7f979,0.0,1,0.0,4.0,NO_PRIORITY,NO_ACTION
4237017,4237018,content_ef5d612068377c3e,0.0,1,0.0,9.0,NO_PRIORITY,NO_ACTION
4237018,4237019,content_6b05a7537d56beb8,0.0,1,0.0,3.0,NO_PRIORITY,NO_ACTION
4237019,4237020,content_3c3e469c8b1708dc,0.0,1,0.0,73.0,NO_PRIORITY,NO_ACTION



Leakage check:
No product flags used.
No future label used.
Only observable search-performance fields were used.

Unexpected columns: set()
Leakage check PASSED.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.